# 03 - Modele Regression Logistique

Ce notebook compare une baseline lineaire brute a une variante mieux conditionnee par preprocessing.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))

from src.donnees import charger_donnees
from src.evaluation import evaluer_modele_cv
from src.modeles import creer_modele_simple, initialiser_grille_tuning, obtenir_definition_modele
from src.pipelines import creer_pipeline_ameliore
from src.resultats import sauvegarder_mesures, sauvegarder_placeholder_tuning


In [ ]:
MODEL_NAME = "regression_logistique"
USE_PCA = True

definition = obtenir_definition_modele(MODEL_NAME)
X, y, label_encoder = charger_donnees()
print(definition["nom_affiche"])
print(definition["hypothese"])
print(f"X: {X.shape}, classes: {len(label_encoder.classes_)}")


In [ ]:
# 1. Baseline simple
modele_simple = creer_modele_simple(MODEL_NAME)
mesures_simple = evaluer_modele_cv(modele_simple, X, y)
sauvegarder_mesures(MODEL_NAME, "simple", mesures_simple)
pd.DataFrame([mesures_simple], index=["simple"])


In [ ]:
# 2. Variante preprocesssee
modele_preprocess = creer_pipeline_ameliore(MODEL_NAME, utiliser_pca=USE_PCA)
mesures_preprocess = evaluer_modele_cv(modele_preprocess, X, y)
mesures_preprocess["utilise_pca"] = USE_PCA
sauvegarder_mesures(MODEL_NAME, "preprocessed", mesures_preprocess)
pd.DataFrame([mesures_simple, mesures_preprocess], index=["simple", "preprocessed"])[["val_accuracy_mean", "val_f1_macro_mean", "elapsed_seconds"]]


In [ ]:
# 3. Boilerplate de tuning
from sklearn.model_selection import GridSearchCV

grille_tuning = initialiser_grille_tuning(MODEL_NAME)
placeholder_tuning = {
    "modele": MODEL_NAME,
    "statut": "boilerplate_only",
    "grille": grille_tuning,
    "note": "Le vrai GridSearchCV sera branche plus tard.",
    "exemple_code": [
        "grid = GridSearchCV(estimator=modele_preprocess, param_grid=grille_tuning, scoring='f1_macro', cv=5, n_jobs=-1)",
        "grid.fit(X, y)",
        "best_params = grid.best_params_",
    ],
}
sauvegarder_placeholder_tuning(MODEL_NAME, placeholder_tuning)
grille_tuning
